# Auditing components.yaml

This notebook runs the AuditSystem against the main components.yaml file to evaluate solution quality.

In [ ]:
from pathlib import Path

import ibis as ib
from ibis import _

from iacs.registrar import Registrar

ib.options.interactive = True  # auto-display results (like pandas)

## Load manifest

Loads both the builtins YAML (component type definitions) and the iacs Python source tree (modules, classes, and functions with docstrings or `__iacs__` metadata).

In [ ]:
import atexit

BUILTINS_DIR = Path.cwd().parent / "builtins"
IACS_SRC_DIR = Path.cwd().parent / "iacs"

a = Registrar.from_manifest([BUILTINS_DIR, IACS_SRC_DIR])
atexit.register(a.registry.close)

## Evaluate highest-priority tasks to work on

In [ ]:
entity_ids = a.get("entity_id")
entity_ids

In [ ]:
parents = a.get("parent")
parents

In [ ]:
reqs = a.get("requirement")
reqs

In [ ]:
priority_product = a.get("priority_product")

# Highest priority leaf requirements (nodes with no children)
(
    reqs
    .join(parents, reqs.entity_id == parents.parent_id, how="anti")
    .join(entity_ids.select(_.value, _.entity_key, _.path), _.entity_id == entity_ids.value)
    .join(priority_product, "entity_id")
    .order_by(ib.desc(_.priority_product))
    .select(_.entity_key, _.priority_product, _.value.name("requirement_type"))
)

In [ ]:
a.get("solution")

In [ ]:
ibis.sql

In [ ]:
cols("alias")

In [ ]:
# Need to use cols to select alias here because alias is also the name of a function in ibis
from ibis.selectors import cols
entity_ids.select(entity_ids.value.name("solution_eid"), solution_alias=cols("alias"))

In [ ]:
a.load_dataflow("audit.requirement_coverage")
a.execute("updated_registry")

In [ ]:
a.registry._con.sql(
    """
select
    rc.*,
    ei.alias,
    ei2.alias as solution_alias
from requirement_coverage rc
join entity_id ei
    on rc.entity_id = ei.value
left join entity_id ei2
    on rc.solution_eid = ei2.value
    """
)

In [ ]:

(
    a.get("requirement_coverage")
    .join(entity_ids.select(entity_id=cols("value"), alias=cols("alias")), "entity_id")
    .join(entity_ids.select(cols("value"), solution_alias=cols("alias")), [("solution_eid", "value")])
)